# **Quickstart notebook to run training on Task Verification Transformer**

This notebook was intended for kaggle, so all paths and settings refers to a kaggle implementation

# **Clone the repository**

In [ ]:
%cd /kaggle/working
!rm -rf AML_error_recognition

!mkdir checkpoints

!git clone --recursive https://github.com/SimoneColu/AML_error_recognition.git

%cd AML_error_recognition
!git checkout task_verification

!git submodule update --init --recursive


# **Install the requirements**

In [ ]:
# Install main requirements quietly
!pip install -q -r requirements.txt

# Fix conflicting versions (uninstall first, then reinstall specific versions)
!pip uninstall -y -q torchaudio torcheval
!pip install -q torcheval==0.0.7 torchaudio==2.1.2

!pip install loguru

In [ ]:

!pip uninstall -y -q torcheval
!pip install -q torcheval==0.0.7

# **Use Wandb API KEY**

**Run this cell after putting your wandb key in kaggle secrets**

In [3]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()


api_key = user_secrets.get_secret("WANDB_API_KEY")

print("WANDB_API_KEY trovata:", api_key is not None)

import os
os.environ["WANDB_API_KEY"] = api_key

WANDB_API_KEY trovata: True


# **Feature loading and preprocess to match the desired format**

In [ ]:
import numpy as np
import os
from tqdm import tqdm

def process_and_aggregate_features(source_folder, output_file_path):
    """
    Reads all .npz files, extracts 'embeddings' as features, parses the filename for ID,
    and aggregates them into a single list of dictionaries.
    
    Output format per sample:
    {
        "video_id": "1_7",       # Parsed from filename
        "features": np.array,    # From 'embeddings' in .npz
        "label": None            # Placeholder
    }
    """
    
    # 1. Get list of files
    if not os.path.exists(source_folder):
        print(f"Error: Source folder '{source_folder}' does not exist.")
        return

    files = [f for f in os.listdir(source_folder) if f.endswith('.npz')]
    print(f"found {len(files)} .npz files. Processing...")

    aggregated_data = []
    
    # 2. Process Loop
    for filename in tqdm(files, desc="Processing"):
        file_path = os.path.join(source_folder, filename)
        
        try:
            # Parse ID: "1_7_video.npz" -> "1_7"
            parts = filename.split('_')
            if len(parts) < 2:
                print(f"Skipping malformed filename: {filename}")
                continue
                
            # Reconstruct the ID (e.g., 1_7)
            current_video_id = f"{parts[0]}_{parts[1]}"
            
            with np.load(file_path) as data:
                if 'embeddings' not in data:
                    print(f"'embeddings' key missing in {filename}")
                    continue
                
                # Extract features
                # Copying ensures it owns the memory and isn't a view of the closed file
                features = data['embeddings'].copy() 
                
                # Create the sample dictionary
                sample = {
                    "video_id": current_video_id,
                    "features": features,
                    "label": -1  # Placeholder
                }
                
                aggregated_data.append(sample)
                
        except Exception as e:
            print(f"Error processing {filename}: {e}")

    # 3. Save to single .npy file
    if aggregated_data:
        print(f"Saving {len(aggregated_data)} samples to {output_file_path}...")
        np_data = np.array(aggregated_data, dtype=object)
        np.save(output_file_path, np_data, allow_pickle=True)
        print("Done!")
    else:
        print("No data was aggregated.")

# --- CONFIGURATION ---
# Folder containing your .npz files
SOURCE_DIR = "/kaggle/input/step-embeddings-768/step_embeddings_768" 

# Where to save the final file (this is what you pass to your train script)
OUTPUT_FILE = "/kaggle/working/recipes_features.npy"

process_and_aggregate_features(SOURCE_DIR, OUTPUT_FILE)


### **Load the checkpoints**

In [25]:
!cp -r "/kaggle/input/frutta2-68" /kaggle/working/checkpoints

### **Reset the checkpoints**

In [30]:
!rm -rf /kaggle/working/checkpoints

# **Run the LOO**

**These are the hyperparameters that we used to train our model**

In [12]:
%cd /kaggle/working/AML_error_recognition

/kaggle/working/AML_error_recognition


In [ ]:
!python train_tv.py \
  --model_name "task_verification_transformer" \
  --recipe_features_path "/kaggle/working/recipes_features.npy"\
  --ckpt_directory /kaggle/working/checkpoints \
  --num_epochs 30 \
  --batch_size 8 \
  --lr 1e-4 \
  --weight_decay 5e-2 \
  --dropout 0.3

##

##

## **Extra: Execute to print results Analytics**

In [ ]:
import torch
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    confusion_matrix, 
    classification_report,
    roc_auc_score  
)

# --- CONFIGURAZIONE ---
CHECKPOINT_DIR = "/kaggle/working/checkpoints/loo_predictions"
OUTPUT_DIR = "/kaggle/working/results_output" 
SAVE_PLOTS = True 

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- LOGGER CLASS TO SAVE TEXT ---
class Logger(object):
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w")
    
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        self.terminal.flush()
        self.log.flush()

# Start recording text output
sys.stdout = Logger(os.path.join(OUTPUT_DIR, "metrics_report.txt"))

# --- MAIN CODE ---
pt_files = glob.glob(os.path.join(CHECKPOINT_DIR, "fold_*_metrics.pt"))
data_list = []

print(f"Reading {len(pt_files)} files...")

for file_path in pt_files:
    try:
        content = torch.load(file_path, map_location='cpu')
        
        row = {
            'fold': content.get('fold'),
            'label': int(content.get('label')),
            'pred_class': int(content.get('pred_class', content.get('pred'))),
            'prob': float(content.get('prob', 0.0)),
            'logits': float(content.get('logits', 0.0)),
            'train_loss_history': list(content.get('train_loss_history', [])),
            'val_loss_history': list(content.get('val_loss_history', []))
        }
        data_list.append(row)
    except Exception as e:
        print(f"Error reading {os.path.basename(file_path)}: {e}")

if data_list:
    df = pd.DataFrame(data_list)
    df = df.sort_values(by='fold').reset_index(drop=True)
    df['CORRECT'] = df['label'] == df['pred_class']

    # SAVE DATA TO CSV
    df.to_csv(os.path.join(OUTPUT_DIR, "results_dataframe.csv"), index=False)
    print(f"Dataframe saved to {OUTPUT_DIR}/results_dataframe.csv")

    # --- 1. METRICHE NUMERICHE ---
    y_true = df['label']
    y_pred = df['pred_class']
    y_prob = df['prob']  

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # --- AUC CALCULATION ---
    try:
        if len(np.unique(y_true)) > 1:
            auc_score = roc_auc_score(y_true, y_prob)
        else:
            auc_score = 0.0
            print("Warning: Only one class present in y_true. AUC set to 0.0")
    except Exception as e:
        print(f"Error calculating AUC: {e}")
        auc_score = 0.0

    print("\n" + "="*40)
    print("            METRICS REPORT            ")
    print("="*40)
    print(f"Total Samples (Folds): {len(df)}")
    print(f"Accuracy:  {accuracy:.2%}")
    print(f"AUC:       {auc_score:.4f}")  # <--- PRINT AUC
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("-" * 40)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    # --- 2. CONFUSION MATRIX ---
    try:
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
        plt.title(f'Confusion Matrix (AUC: {auc_score:.2f})') # Added AUC to title
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        
        if SAVE_PLOTS: 
            save_path = os.path.join(OUTPUT_DIR, 'confusion_matrix.png')
            plt.savefig(save_path, bbox_inches='tight')
            print(f"Plot saved: {save_path}")
            
        plt.show()
    except Exception as e:
        print(f"Error plotting confusion matrix: {e}")

    # --- 3. LOSS CURVES (AVERAGE) ---
    def get_avg_loss_curve(history_column):
        histories = [h for h in df[history_column] if isinstance(h, list) and len(h) > 0]
        if not histories: return None, None, None
        
        max_len = max(len(h) for h in histories)
        matrix = np.full((len(histories), max_len), np.nan)
        
        for i, h in enumerate(histories):
            matrix[i, :len(h)] = h
            
        means = np.nanmean(matrix, axis=0)
        stds = np.nanstd(matrix, axis=0)
        epochs = np.arange(1, max_len + 1)
        
        return epochs, means, stds

    t_epochs, t_means, t_stds = get_avg_loss_curve('train_loss_history')
    v_epochs, v_means, v_stds = get_avg_loss_curve('val_loss_history')

    # --- PRINT LOSS STATISTICS ---
    print("\n" + "="*40)
    print("            LOSS STATISTICS            ")
    print("="*40)
    
    if t_means is not None and len(t_means) > 0:
        print(f"Final Avg Train Loss: {t_means[-1]:.4f}")
    
    if v_means is not None and len(v_means) > 0:
        print(f"Final Avg Val Loss:   {v_means[-1]:.4f}")
    else:
        print("No validation loss history found.")
    print("-" * 40)

    if t_epochs is not None:
        plt.figure(figsize=(10, 6))
        
        plt.plot(t_epochs, t_means, label='Avg Train Loss', color='blue')
        plt.fill_between(t_epochs, t_means - t_stds, t_means + t_stds, color='blue', alpha=0.15)
        
        if v_epochs is not None:
            plt.plot(v_epochs, v_means, label='Avg Val Loss', color='orange', linestyle='--')
            plt.fill_between(v_epochs, v_means - v_stds, v_means + v_stds, color='orange', alpha=0.15)
        
        plt.title('Average Loss Curve across all Folds')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        if SAVE_PLOTS: 
            save_path = os.path.join(OUTPUT_DIR, 'avg_loss_curve.png')
            plt.savefig(save_path, bbox_inches='tight')
            print(f"Plot saved: {save_path}")

        plt.show()
    else:
        print("\nNo loss history found to generate plots.")

    # --- 4. ANALISI ERRORI ---
    total_errors = len(df) - df['CORRECT'].sum()
    if total_errors > 0:
        print(f"\n--- DETTAGLIO ERRORI ({total_errors}) ---")
        df['min_loss'] = df['train_loss_history'].apply(lambda x: min(x) if x else None)
        cols = ['fold', 'label', 'pred_class', 'prob', 'min_loss']
        print(df[~df['CORRECT']][cols])

else:
    print("No valid data found.")

# Check if the current stdout is actually my custom Logger
# The logger takes the stdout so we have to restor it
if hasattr(sys.stdout, 'terminal'):
    sys.stdout = sys.stdout.terminal
    print("Output restored to normal!")
else:
    print("Output was already normal (or I couldn't find the backup stream).")

## **Execute to zip the results**

In [ ]:
import shutil
import os
from IPython.display import FileLink, display

# --- 1. SETUP ---
DIR_TO_ZIP = "/kaggle/working/results_output" 
OUTPUT_FILENAME = "results_output" 
WORKING_DIR = "/kaggle/working"

# --- 2. ZIP CREATION ---
print(f"Zipping {DIR_TO_ZIP}...")

if not os.path.exists(DIR_TO_ZIP):
    print(f"Error: Folder {DIR_TO_ZIP} not found. Cannot zip.")
else:
    shutil.make_archive(
        base_name=os.path.join(WORKING_DIR, OUTPUT_FILENAME), 
        format="zip", 
        root_dir=DIR_TO_ZIP
    )
    print("Zip created.")

# --- 3. GENERATE LINK ---
os.chdir(WORKING_DIR)

expected_file = f"{OUTPUT_FILENAME}.zip"

if os.path.exists(expected_file):
    print(f"\nFile found: {expected_file}")
    print(f"Size: {os.path.getsize(expected_file) / 1e6:.2f} MB")
    print("Click below to download:")
    display(FileLink(expected_file))
else:
    print(f"Error: Could not find {expected_file}")
    print("Files currently in folder:")
    print(os.listdir(WORKING_DIR))

## **Execute to zip checkpoints**

In [ ]:
import shutil
import os

DIR_DA_SALVARE = "/kaggle/working/checkpoints"  
ZIP_PATH = "/kaggle/working/checkpoints.zip"

print("Zippando:", DIR_DA_SALVARE)
print("Esiste?", os.path.exists(DIR_DA_SALVARE))

if not os.path.exists(DIR_DA_SALVARE):
    raise RuntimeError("La directory NON esiste")

# Cancella zip precedente se esiste
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

shutil.make_archive(
    base_name="/kaggle/working/checkpoints",
    format="zip",
    root_dir=DIR_DA_SALVARE
)

print("ZIP creato?")
!ls -lh /kaggle/working

